In [14]:
import numpy as np
from astropy.cosmology import FlatLambdaCDM
from astropy.cosmology import z_at_value
from astropy import units as u
from scipy.integrate import quad
from scipy.interpolate import interp1d

# Compute redshift of source planes behind lens planes
# Load lens redshifts:

zfile=np.loadtxt("z2ts.txt",delimiter=',')
z_list=np.flip(zfile[1:58,0])



In [15]:
H0 = 71.0
Om0=0.2648
Ob0=0.044792699861138666
ncosmo = FlatLambdaCDM(H0=H0, Om0=Om0, Ob0=Ob0)


In [17]:
zfile=np.loadtxt("z2ts.txt",delimiter=',')
snaplist = zfile[0:58,1].astype(int)
zsnap=zfile[0:58,0]
print(snaplist[0:58])
#print(zsnap)
dllow = np.empty((len(zsnap), 2), dtype=np.float)
dlup  = np.empty_like(dllow)
dllen = np.empty_like(dllow)


# construct the lens plane boundaries from SkySim5000, with shells 114 Mpc thick (=  114*h Mpc/h) 
for plane in range(0,58):
    dllow[plane,1] = plane*114.#*ncosmo.h
    dlup[plane,1] = (plane+1)*114.#*ncosmo.h
    #print(plane, dllow[plane,1], dlup[plane,1])

    dllow[plane,0] = z_at_value(ncosmo.comoving_distance, dllow[plane,1]*u.Mpc) if dllow[plane,1] > 0.0 else 0.0
    dlup[plane,0]  = z_at_value(ncosmo.comoving_distance,  dlup[plane,1]*u.Mpc)

    # could use the exact z edge values provided by the z2ts.txt file, instead of the z_at_value calculation, 
    # which gives slightly different answers.
    
# Computing the Lens Planes (Distance Average)
for i, (d1, d2) in enumerate(zip(dllow, dlup)):

    dllen[i, 0]  = quad( lambda z: z * ncosmo.comoving_distance(z).value, dllow[i, 0], dlup[i, 0] )[0]
    dllen[i, 0] /= quad( lambda z:     ncosmo.comoving_distance(z).value, dllow[i, 0], dlup[i, 0] )[0]
    dllen[i, 1]  = ncosmo.comoving_distance(dllen[i, 0]).value

print( "Plane list:     Dl_           Lens             Dl^      [all units in Mpc, not Mpc/h]"  )


for d1, dl, d2 in zip(dllow[:,1], dllen[:,1], dlup[:,1]):

    print("           {0: >10.4f}     {1: >10.4f}      {2: >10.4f} ".format(d1, dl, d2 ))


    
    
#for plane in range(0,29):
#    print(dlup[plane,0], dllow[plane,0])
print('***')
    
rds_upper = np.zeros_like(dlup[:,0])
rds_lower = np.zeros_like(dlup[:,0])


for plane in range(0,58):
    rds_upper[plane], rds_lower[plane] = [dlup[57-plane,0], dllow[57-plane,0]]

#-----
# Get lens redshifts, either from geometry:
zl_list = (3.0/4.0)*(rds_upper**4-rds_lower**4)/(rds_upper**3-rds_lower**3)
#... or from the partice distribution:
tmp=np.loadtxt("z_or_skysim.txt")
zlens_exact=np.zeros_like(tmp)
for plane in range(0,57):
    zlens_exact[plane]=tmp[56-plane]
zlens_exact = zlens_exact[:57] 
    
print( "Plane list:     z^       z_               zl_geo             zl_file             zl_ana"  )
for plane in range(0,57):
    print(rds_upper[plane], rds_lower[plane], zl_list[plane],zlens_exact[plane], dllen[57-plane, 0] )


[499 487 475 464 453 442 432 421 411 401 392 382 373 365 355 347 338 331
 323 315 307 300 293 286 279 272 266 253 247 241 235 230 224 219 213 208
 203 198 194 189 184 180 176 171 167 163 159 155 151 148 144 141 137 134
 131 127 124 121]


/tmp/ipykernel_2057/2042953706.py:6: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  dllow = np.empty((len(zsnap), 2), dtype=np.float)


Plane list:     Dl_           Lens             Dl^      [all units in Mpc, not Mpc/h]
               0.0000        76.1044        114.0000 
             114.0000       177.4902        228.0000 
             228.0000       288.9674        342.0000 
             342.0000       401.8895        456.0000 
             456.0000       515.2937        570.0000 
             570.0000       628.9170        684.0000 
             684.0000       742.6584        798.0000 
             798.0000       856.4707        912.0000 
             912.0000       970.3288       1026.0000 
            1026.0000      1084.2183       1140.0000 
            1140.0000      1198.1303       1254.0000 
            1254.0000      1312.0588       1368.0000 
            1368.0000      1426.0000       1482.0000 
            1482.0000      1539.9509       1596.0000 
            1596.0000      1653.9097       1710.0000 
            1710.0000      1767.8747       1824.0000 
            1824.0000      1881.8448       1938.00